# 2024 Game Preprocessing



In [1]:
!pip install pybaseball

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.5/416.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.7/856.7 kB 51.7 MB/s eta 0:00:00


In [2]:
# Import necessary modules
from google.colab import drive
import sys

# Mount Google Drive to access files stored in your Google Drive
drive.mount('/content/drive')

# NOTE: Update the paths below to match the location of your project files in Google Drive.
# Replace with your own directory if different.

# Add the main 'utils' directory to Python's module search path
# This allows you to import custom utility modules from this folder
path = "/content/drive/MyDrive/Homerun Prediction Model/utils"
sys.path.append(path)


ValueError: mount failed

In [ ]:
from preprocessing import add_batter_names
from preprocessing import add_batter_full_name
from preprocessing import prepare_barrels

from pybaseball import statcast
from pybaseball import playerid_reverse_lookup
import numpy as np
import pandas as pd
import os

### Retrieving Statcast Data for the 2024 MLB Season

In [ ]:
season_2024 = statcast(start_dt="2024-03-20", end_dt="2024-09-30")

In [ ]:
# Displaying first ten rows
season_2024.head(10)

#### Changing `player_name` to `pitcher_name` and `batter` to `batter_id`



In [ ]:
season_2024 = season_2024.rename(columns={
    "player_name": "pitcher_name",
    "batter": "batter_id"
})

## Mapping Batter IDs to Player Names

Although not necessary for running the model, I will match the `batter` ID to the player's name. This will allow us to predict individual batter homerun probabilities. This can be done using the `playerid_reverse_lookup` function, which returns a DataFrame containing the player ID (`key_mlbam`) along with the batter's first and last name.


In [ ]:
# Add batter names to the 2024 season dataset
season_2024 = add_batter_names(season_2024)
season_2024.head(10)

### Creating a Full Name Column for Batters

To streamline player identification in the dataset, this step standardizes the formatting of first and last names by capitalizing them and then concatenates them into a new `full_name` column.

In [ ]:
season_2024 = add_batter_full_name(season_2024)
season_2024

## Barrels


Now we can calculate barrels, defined here as cases where `launch_speed_angle` equals `6`. Before doing so, we first filter the dataset to keep only the final pitch of each at-bat, since the at-bat outcome (such as a home run) is determined on that pitch. This makes the at-bat our unit of observation and ensures we are focusing on the decisive moment. Finally, we create binary indicators to capture whether the last pitch was a barrel and whether it resulted in a home run.


In [ ]:
at_bats_2024 = prepare_barrels(season_2024)
at_bats_2024.head(10)

### Function: `compute_group_mean_with_overall`

This function calculates the mean of a column for each group and also adds one extra row at the top showing the overall mean across the whole dataset.

In [ ]:
def compute_group_mean_with_overall(
    df: pd.DataFrame,
    group_col: str,
    value_col: str = "barrel",
    overall_id: str = "overall",
    mean_col_name: str | None = None
) -> pd.DataFrame:
    """
    Compute per-group mean of `value_col` plus a first row for the true overall mean.

    Parameters
    ----------
    df : pd.DataFrame
        Input data.
    group_col : str
        Column to group by (e.g., 'batter_id', 'pitcher_name').
    value_col : str, default 'barrel'
        Column whose mean is computed.
    overall_id : str, default 'overall'
        Label used for the overall row in `group_col`.
    mean_col_name : str | None
        Optional name of the output mean column; if None, uses f"{value_col}_mean".

    Returns
    -------
    pd.DataFrame
        DataFrame with an overall row first, followed by per-group means.
    """
    if mean_col_name is None:
        mean_col_name = f"{value_col}_mean"

    # Per-group mean
    per_group = (
        df.groupby(group_col, dropna=False)[value_col]
          .mean()
          .reset_index()
          .rename(columns={value_col: mean_col_name})
    )

    # True overall mean across all rows (not mean of means)
    overall_mean = df[value_col].mean()
    overall_row = pd.DataFrame({group_col: [overall_id], mean_col_name: [overall_mean]})

    # Combine with overall first
    out = pd.concat([overall_row, per_group], ignore_index=True)
    return out

#### Batter Barrel Mean

In [ ]:
# Batters (2024)
batter_barrel_means_2024 = compute_group_mean_with_overall(
    at_bats_2024, group_col="batter_id",
    value_col="barrel",
    mean_col_name="batter_mean_barrel_rate_2024"
)

batter_barrel_means_2024.head(10)

#### Pitcher Barrel Mean

In [ ]:
# Pitchers (2024) – mean barrel rate allowed per pitcher
pitcher_barrel_means_2024 = compute_group_mean_with_overall(
    at_bats_2024, group_col="pitcher_name",
    value_col="barrel",
    mean_col_name="pitcher_mean_barrel_rate_allowed_2024"
)

pitcher_barrel_means_2024.head(10)

## Export

Save the batter and pitcher barrel means from 2024 to Google Drive in CSV format.


In [ ]:
# Define full path to your project data folder
drive_path = "/content/drive/MyDrive/Homerun Prediction Model/data"

# Export batter means
batter_barrel_means_2024.to_csv(
    os.path.join(drive_path, "batter_barrel_means_2024.csv"), index=False
)

# Export pitcher means
pitcher_barrel_means_2024.to_csv(
    os.path.join(drive_path, "pitcher_barrel_means_2024.csv"), index=False
)